# XD-Violence: Video-Level Violence Classification

This notebook trains a visual baseline on the seven folder labels using actual frames decoded from the MP4 clips. It preserves the dataset?s official `train` and `test` directories. The baseline samples eight frames per clip, applies pretrained EfficientNet-B0 to each frame, and averages frame logits for one video-level prediction.

The full dataset contains 4,464 video clips (expected dataset size). Run with a GPU for practical training time.

In [ ]:
from pathlib import Path
import os
import re
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score, f1_score

SEED = 42
NUM_FRAMES = 8
IMAGE_SIZE = 224
BATCH_SIZE = 8  # each item contains 8 frames; lower this if GPU memory is limited
NUM_EPOCHS = 5
NUM_WORKERS = 2

np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Kaggle dataset location. Change this only if your mounted dataset uses another path.
DATASET_PATH = Path("/kaggle/input/datasets/bypktt/xd-violence")
if not DATASET_PATH.exists():
    # Kaggle sometimes mounts the same dataset under a versioned directory.
    candidates = list(Path("/kaggle/input").glob("**/train")) if Path("/kaggle/input").exists() else []
    candidates = [path.parent for path in candidates if (path.parent / "test").is_dir()]
    if len(candidates) == 1:
        DATASET_PATH = candidates[0]
    else:
        raise FileNotFoundError(
            f"Dataset root not found at /kaggle/input/datasets/bypktt/xd-violence. "
            "Set DATASET_PATH to the directory containing train/ and test/."
        )
print("Dataset root:", DATASET_PATH)

## Build the official train/test manifest

Folder names are the target labels. The manifest only reads the existing `train` and `test` directories and never creates a random split.

In [ ]:
records = []
for split in ["train", "test"]:
    split_dir = DATASET_PATH / split
    if not split_dir.is_dir():
        raise FileNotFoundError(split_dir)
    for class_dir in sorted(split_dir.iterdir()):
        if not class_dir.is_dir():
            continue
        for video_path in sorted(class_dir.glob("*.mp4")):
            records.append({"video_path": str(video_path), "label": class_dir.name, "split": split})

df = pd.DataFrame(records)
if df.empty:
    raise RuntimeError(f"No MP4 files found under {DATASET_PATH}/train and /test")
print("Total videos:", len(df))
print("\nClass distribution:")
print(df["label"].value_counts().sort_index())
print("\nSplit distribution:")
print(df["split"].value_counts())
print("\nClass-wise split distribution:")
display(pd.crosstab(df["label"], df["split"], margins=True))
assert df["video_path"].is_unique
assert df["label"].notna().all()
assert df["split"].isin(["train", "test"]).all()
assert set(df["label"].unique()) == {"Normal", "Riot", "Shooting", "Explosion", "Fighting", "Abuse", "CarAccident"}, "Check folder labels; expected seven XD-Violence classes."
print("Manifest successfully created.")

## Check folder labels against filename annotations

Filename annotation codes may have a different naming scheme from the class folders. This is a diagnostic only; the model target remains the folder label.

In [ ]:
def extract_filename_label(path):
    filename = os.path.basename(path)
    match = re.search(r"_label_([A-Za-z0-9]+)", filename)
    return match.group(1) if match else None

df["filename_label"] = df["video_path"].apply(extract_filename_label)
print("Folder label vs filename annotation:")
display(pd.crosstab(df["label"], df["filename_label"], margins=True, dropna=False))

## Decode and sample video frames

OpenCV reads eight uniformly spaced frames from each video. Frames are converted from BGR to RGB and resized for EfficientNet-B0.

In [ ]:
def extract_frames(video_path, num_frames=NUM_FRAMES, image_size=IMAGE_SIZE):
    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        return []

    indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        success, frame = cap.read()
        if not success:
            continue
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (image_size, image_size), interpolation=cv2.INTER_AREA)
        frames.append(frame)
    cap.release()
    return frames

sample_path = df.iloc[0]["video_path"]
sample_frames = extract_frames(sample_path)
print("Video:", sample_path)
print(f"Frames extracted: {len(sample_frames)} / {NUM_FRAMES}")
if not sample_frames:
    raise RuntimeError(f"Could not decode sample video: {sample_path}")
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax in axes.flat:
    ax.axis("off")
for ax, frame in zip(axes.flat, sample_frames):
    ax.imshow(frame)
plt.suptitle("Uniformly sampled video frames")
plt.tight_layout()
plt.show()

## Create video-level datasets and EfficientNet-B0

Each dataset item is a complete video tensor with shape `(frames, channels, height, width)`. EfficientNet classifies each sampled frame, and the model averages its frame logits before calculating one loss per video.

In [ ]:
class VideoFrameDataset(Dataset):
    def __init__(self, manifest, class_to_idx, num_frames=NUM_FRAMES):
        self.manifest = manifest.reset_index(drop=True)
        self.class_to_idx = class_to_idx
        self.num_frames = num_frames
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.manifest)

    def __getitem__(self, index):
        row = self.manifest.iloc[index]
        frames = extract_frames(row["video_path"], self.num_frames)
        if not frames:
            raise RuntimeError(f"Could not decode video: {row['video_path']}")
        # Keep tensor size fixed if a damaged/short clip yields fewer decoded frames.
        while len(frames) < self.num_frames:
            frames.append(frames[-1].copy())
        frames = frames[:self.num_frames]
        video = torch.stack([self.transform(Image.fromarray(frame)) for frame in frames])
        return video, self.class_to_idx[row["label"]]

class_names = sorted(df["label"].unique())
class_to_idx = {name: idx for idx, name in enumerate(class_names)}
idx_to_class = {idx: name for name, idx in class_to_idx.items()}
train_df = df[df["split"] == "train"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)
train_dataset = VideoFrameDataset(train_df, class_to_idx)
test_dataset = VideoFrameDataset(test_df, class_to_idx)
loader_options = {"num_workers": NUM_WORKERS, "pin_memory": DEVICE.type == "cuda"}
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, **loader_options)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_options)

weights = models.EfficientNet_B0_Weights.DEFAULT
model = models.efficientnet_b0(weights=weights)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(class_names))
model = model.to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
print("Classes:", class_names)
print(f"Train videos: {len(train_dataset):,} | Test videos: {len(test_dataset):,}")

## Train on the official training set

The official test set is reserved for final evaluation. Training updates EfficientNet using video-level predictions formed by averaging the sampled frame logits.

In [ ]:
history = []
for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    train_correct = 0
    train_count = 0
    for videos, labels in train_loader:
        # videos: [batch, frames, channels, height, width]
        videos = videos.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        batch_size, n_frames, channels, height, width = videos.shape
        optimizer.zero_grad(set_to_none=True)
        frame_logits = model(videos.reshape(batch_size * n_frames, channels, height, width))
        video_logits = frame_logits.reshape(batch_size, n_frames, -1).mean(dim=1)
        loss = criterion(video_logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * batch_size
        train_correct += (video_logits.argmax(dim=1) == labels).sum().item()
        train_count += batch_size
    epoch_loss = running_loss / train_count
    epoch_accuracy = train_correct / train_count
    history.append({"epoch": epoch + 1, "loss": epoch_loss, "accuracy": epoch_accuracy})
    print(f"Epoch {epoch + 1}/{NUM_EPOCHS} ? loss: {epoch_loss:.4f} ? train video accuracy: {epoch_accuracy:.4f}")

history_df = pd.DataFrame(history)
display(history_df)

## Evaluate video-level predictions on the official test set

In [ ]:
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for videos, labels in test_loader:
        videos = videos.to(DEVICE, non_blocking=True)
        batch_size, n_frames, channels, height, width = videos.shape
        frame_logits = model(videos.reshape(batch_size * n_frames, channels, height, width))
        video_logits = frame_logits.reshape(batch_size, n_frames, -1).mean(dim=1)
        y_pred.extend(video_logits.argmax(dim=1).cpu().numpy())
        y_true.extend(labels.numpy())

print(f"Official test video accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"Weighted F1: {f1_score(y_true, y_pred, average='weighted'):.4f}")
print(classification_report(y_true, y_pred, labels=list(range(len(class_names))), target_names=class_names, zero_division=0))
cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(9, 8))
display.plot(ax=ax, xticks_rotation=45, cmap="Blues", colorbar=False)
plt.title("EfficientNet-B0 video-level test confusion matrix")
plt.tight_layout()
plt.show()

## Next step

This is a frame-based visual baseline: EfficientNet-B0 learns from actual pixels, and mean frame logits provide a simple video-level decision. Temporal architectures such as VideoMAE or a 3D CNN can be explored after establishing this baseline.